# The "do it wrong" section: `exposure` vs. `treatment`

`treatment` is the randomized coin flip (eligibility) — the correct variable for a causal estimate. `exposure` is not randomized: Criteo's auction system selects who actually sees an ad based on predicted value, so the exposed group is pre-loaded with users who'd likely convert anyway.

This notebook quantifies the bias from making that mistake: rerun `compute_ate` with `treatment_col="exposure"` instead of `"treatment"`, on the same data, and compare.

In [ ]:
import sys
sys.path.insert(0, "../src")

from uplift.data import load_sample
from uplift.ate import compute_ate

df = load_sample()

results = {}
for outcome in ["visit", "conversion"]:
    correct = compute_ate(df, outcome, treatment_col="treatment")
    naive = compute_ate(df, outcome, treatment_col="exposure")
    results[outcome] = (correct, naive)

    print(f"=== {outcome} ===")
    print(f"Correct (treatment):  ATE={correct.ate:.5f}  (95% CI: {correct.ci_low:.5f} to {correct.ci_high:.5f})  relative lift={correct.relative_lift:.1%}")
    print(f"Naive (exposure):     ATE={naive.ate:.5f}  (95% CI: {naive.ci_low:.5f} to {naive.ci_high:.5f})  relative lift={naive.relative_lift:.1%}")
    print(f"Naive estimate is {naive.ate / correct.ate:.1f}x the correct ATE")
    print()

## Conclusion

| outcome | correct ATE (treatment) | naive ATE (exposure) | bias multiple |
|---|---|---|---|
| `visit` | 0.01081 (28.6% relative lift) | 0.37918 (1071% relative lift) | **35.1x** |
| `conversion` | 0.00111 (55.3% relative lift) | 0.05308 (3976% relative lift) | **47.8x** |

The bias isn't a modest overestimate — it's a 35-48x inflation. Mechanism: only ~3.6% of eligible users are ever exposed, and Criteo's auction algorithm chooses that narrow slice specifically to maximize predicted value/likelihood-to-convert. The narrower and more aggressive the selection filter, the more extreme the resulting group looks relative to the general population — comparing "the algorithm's top 3.6% pick" against "everyone else" is comparing an extreme outlier slice to baseline, not two roughly-similar groups.

Practical red flag independent of causal reasoning: relative lift figures over 1000% should themselves raise suspicion in any real analysis, before even knowing to check whether the grouping variable was actually randomized.

**This is the headline result for the "why numbers might be wrong" section of the write-up**: it's a concrete, quantified demonstration of exactly the mistake the whole randomization argument (notebooks 01-02) was built to prevent.